## Imports og tilkobling

In [2]:
import sys
import os
sys.path.append('../../../src')
import feature_engineering.værdata as vd
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient

load_dotenv()

connection_string = os.getenv("AZURE_STORAGE_CONNECTION_STRING")

blob_service_client = BlobServiceClient.from_connection_string(connection_string)
container_name = "processeddata"

## Finn og velg værstasjon

In [3]:
# Breive koordinater
breive_coords = (59.57624, 7.28038)

# Velg stasjon
nearest_station, stations_df = vd.finn_naermeste_stasjon(*breive_coords, stasjon_index=0)
print("Henter data fra stasjon:", nearest_station)

Henter data fra stasjon: SN40880


## Definer periode

In [4]:
vinter_periods = [
    ("2024-11-01T00:00:00Z", "2024-11-30T23:00:00Z"),
    ("2024-12-01T00:00:00Z", "2024-12-31T23:00:00Z"),
    ("2025-01-01T00:00:00Z", "2025-01-31T23:00:00Z"),
    ("2025-11-01T00:00:00Z", "2025-11-30T23:00:00Z"),
    ("2025-12-01T00:00:00Z", "2025-12-31T23:00:00Z"),
    ("2026-01-01T00:00:00Z", "2026-01-20T22:00:00Z")
]

## Hent time for time værdata

In [5]:
weather_df_hourly = vd.fetch_weather_periods_hourly(nearest_station, vinter_periods)
weather_df_hourly.head()

,timestamp,air_temperature,wind_speed,precipitation_mm
0,2024-11-01 00:00:00,7.9,8.2,0.6
1,2024-11-01 01:00:00,7.6,6.1,1.3
2,2024-11-01 02:00:00,7.6,6.9,2.2
3,2024-11-01 03:00:00,5.9,8.4,2.8
4,2024-11-01 04:00:00,4.5,6.5,1.1


## Last opp til Azure Blob

In [5]:
import io

# --- Lag CSV i minnet ---
csv_buffer = io.BytesIO()
weather_df_hourly.to_csv(csv_buffer, index=False)
csv_buffer.seek(0)  # gå til starten av bufferet

# --- Last opp til blob ---
output_blob_name = "BREIVE_weather.csv"
blob_client = blob_service_client.get_blob_client(container=container_name, blob=output_blob_name)
blob_client.upload_blob(csv_buffer, overwrite=True)  # overwrite=True overskriver eksisterende fil

print(f"{output_blob_name} lastet opp til blob!")

BREIVE_weather.csv lastet opp til blob!


# Kvalitetssikring

## Sjekk struktur

In [6]:
print(weather_df_hourly.info())
print(weather_df_hourly.head())

<class 'pandas.DataFrame'>
RangeIndex: 4145 entries, 0 to 4144
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   timestamp         4145 non-null   str    
 1   air_temperature   4145 non-null   float64
 2   wind_speed        4145 non-null   float64
 3   precipitation_mm  4145 non-null   float64
dtypes: float64(3), str(1)
memory usage: 129.7 KB
None
             timestamp  air_temperature  wind_speed  precipitation_mm
0  2024-11-01 00:00:00              7.9         8.2               0.6
1  2024-11-01 01:00:00              7.6         6.1               1.3
2  2024-11-01 02:00:00              7.6         6.9               2.2
3  2024-11-01 03:00:00              5.9         8.4               2.8
4  2024-11-01 04:00:00              4.5         6.5               1.1


## Sjekk tidsserien

In [7]:
weather_df_hourly = weather_df_hourly.sort_values("timestamp")

In [12]:
import pandas as pd

def check_missing_per_period(df, periods):
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"]).dt.tz_localize(None)
    
    for start, end in periods:
        start = pd.to_datetime(start).tz_localize(None)
        end = pd.to_datetime(end).tz_localize(None)

        expected = pd.date_range(start=start, end=end, freq="h")
        
        actual = df[
            (df["timestamp"] >= start) & 
            (df["timestamp"] <= end)
        ]["timestamp"]
        
        missing = expected.difference(actual)
        
        print(f"\nPeriode: {start} → {end}")
        print("Manglende timer:", len(missing))

In [13]:
check_missing_per_period(weather_df_hourly, vinter_periods)


Periode: 2024-11-01 00:00:00 → 2024-11-30 23:00:00
Manglende timer: 1

Periode: 2024-12-01 00:00:00 → 2024-12-31 23:00:00
Manglende timer: 1

Periode: 2025-01-01 00:00:00 → 2025-01-31 23:00:00
Manglende timer: 1

Periode: 2025-11-01 00:00:00 → 2025-11-30 23:00:00
Manglende timer: 1

Periode: 2025-12-01 00:00:00 → 2025-12-31 23:00:00
Manglende timer: 1

Periode: 2026-01-01 00:00:00 → 2026-01-20 22:00:00
Manglende timer: 1


## Sjekk verdier

In [14]:
# Temperatur
weather_df_hourly["air_temperature"].describe()

count    4145.000000
mean       -4.268347
std         7.706626
min       -31.700000
25%        -8.500000
50%        -2.200000
75%         1.400000
max         9.200000
Name: air_temperature, dtype: float64

In [15]:
# Sjekk for ekstreme temperaturer
weather_df_hourly[
    (weather_df_hourly["air_temperature"] < -40) |
    (weather_df_hourly["air_temperature"] > 40)
]

,timestamp,air_temperature,wind_speed,precipitation_mm


In [16]:
# Vind
weather_df_hourly["wind_speed"].describe()

count    4145.000000
mean        2.417949
std         1.922640
min         0.100000
25%         1.000000
50%         1.700000
75%         3.300000
max        13.800000
Name: wind_speed, dtype: float64

In [17]:
# Sjekk for ekstreme vindhastigheter
weather_df_hourly[
    weather_df_hourly["wind_speed"] > 40
]

,timestamp,air_temperature,wind_speed,precipitation_mm


In [18]:
# Nedbør
weather_df_hourly["precipitation_mm"].describe()

count    4145.000000
mean        0.123595
std         0.384155
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         5.800000
Name: precipitation_mm, dtype: float64

In [19]:
# Sjekk for ekstrem nedbør
print("Max:", weather_df_hourly["precipitation_mm"].max())

# Sjekk hvor mange timer det er nedbør
print("Non-zero:", (weather_df_hourly["precipitation_mm"] > 0).sum())

Max: 5.8
Non-zero: 962


## Manglende verdier

In [20]:
weather_df_hourly.isna().sum()

timestamp           0
air_temperature     0
wind_speed          0
precipitation_mm    0
dtype: int64

In [21]:
weather_df_hourly['precipitation_mm'].value_counts(dropna=False)

precipitation_mm
0.0    3183
0.1     313
0.2     135
0.3     114
0.4      65
0.6      43
0.5      41
0.9      29
0.7      28
0.8      26
1.0      24
1.1      21
1.3      18
1.2      16
1.4      14
1.5      10
1.6       8
2.0       6
2.3       5
1.7       5
2.7       4
3.1       4
2.6       4
1.9       4
2.2       3
2.8       3
2.5       3
3.4       3
1.8       3
2.4       3
3.2       1
4.3       1
3.0       1
2.1       1
5.8       1
3.7       1
3.3       1
Name: count, dtype: int64

## Korrelasjon sanity check

In [22]:
weather_df_hourly.corr(numeric_only=True)

,air_temperature,wind_speed,precipitation_mm
air_temperature,1.000000,0.322672,0.204502
wind_speed,0.322672,1.000000,0.316854
precipitation_mm,0.204502,0.316854,1.000000


## Kvalitetsscore

In [23]:
def quality_report(df):
    print("---- QUALITY REPORT ----")
    print("Rows:", len(df))
    print("\nMissing values:")
    print(df.isna().sum())
    
    print("\nNon-zero precipitation:", (df["precipitation_mm"] > 0).sum())
    print("Max precipitation:", df["precipitation_mm"].max())
    
    print("\nTemperature range:",
          df["air_temperature"].min(),
          "to",
          df["air_temperature"].max())
    
    print("\nWind max:", df["wind_speed"].max())

quality_report(weather_df_hourly)

---- QUALITY REPORT ----
Rows: 4145

Missing values:
timestamp           0
air_temperature     0
wind_speed          0
precipitation_mm    0
dtype: int64

Non-zero precipitation: 962
Max precipitation: 5.8

Temperature range: -31.7 to 9.2

Wind max: 13.8
